# Seoul Commercial Area Industry Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justin45359-hue/seoul-commercial-area-analysis/blob/main/seoul_commercial_analysis_colab_direct.ipynb)

**Purpose:** Identify industries with high sales and strong local concentration across Seoul commercial-area types using 2025 estimated-sales data.

The analysis combines annual sales, transaction counts, industry concentration (LQ), and quarterly sales trends.


In [ ]:
from google.colab import files

uploaded = files.upload()


In [ ]:
# Seoul Commercial Area Industry Analysis

import matplotlib.pyplot as plt


# Use the file uploaded in the previous cell.
DATA_FILE = list(uploaded.keys())[0]


# English labels used in charts.
market_name_en = {
    "노량진역(노량진)": "Noryangjin Fish Market",
    "가산디지털단지": "Gasan Digital Complex",
    "용산전자상가(용산역)": "Yongsan Electronics Market",
    "가락시장": "Garak Market",
    "잠실역": "Jamsil Station",
    "강남역": "Gangnam Station",
    "여의도역": "Yeouido Station",
    "종로3가역": "Jongno 3-ga Station",
    "목동신시가지": "Mok-dong New Town",
    "성수동카페거리": "Seongsu Cafe Street",
    "청량리청과물시장": "Cheongnyangni Fruit Market",
    "청량리종합시장": "Cheongnyangni General Market",
    "청량리수산시장": "Cheongnyangni Fish Market",
    "독산동 우시장": "Doksan Cattle Market",
    "명동·남대문 관광특구": "Myeongdong-Namdaemun Tourist Area",
    "이태원 관광특구": "Itaewon Tourist Area",
    "동대문패션타운 관광특구": "Dongdaemun Fashion Town",
    "잠실 관광특구": "Jamsil Tourist Area",
    "영등포시장역 3번": "Yeongdeungpo Market Station",
    "한강로동 땡땡거리": "Hangang-ro Street",
    "도곡초등학교": "Dogok Elementary School",
    "방이역 4번": "Bangyi Station"
}

industry_name_en = {
    "수산물판매": "Seafood Sales",
    "일반의류": "General Apparel",
    "컴퓨터및주변장치판매": "Computers and Peripherals",
    "컴퓨터·주변장치판매": "Computers and Peripherals",
    "반찬가게": "Side Dish Stores",
    "가전제품": "Home Appliances",
    "시계및귀금속": "Watches and Jewelry",
    "시계·귀금속": "Watches and Jewelry",
    "일반의원": "General Clinics",
    "한식음식점": "Korean Restaurants",
    "일반교습학원": "Private Academies",
    "청과상": "Fruit and Vegetable Stores",
    "육류판매": "Meat Sales",
    "화장품": "Cosmetics",
    "양식음식점": "Western Restaurants",
    "신발": "Footwear",
    "완구": "Toys",
    "조명용품": "Lighting Products"
}

market_type_en = {
    "발달상권": "Commercial Districts",
    "전통시장": "Traditional Markets",
    "관광특구": "Tourist Areas",
    "골목상권": "Neighborhood Commercial Areas"
}


def english_market_name(market_name):
    if market_name in market_name_en:
        return market_name_en[market_name]
    return market_name


def english_industry_name(industry_name):
    if industry_name in industry_name_en:
        return industry_name_en[industry_name]
    return industry_name


def bar_label(market_name, industry_name):
    return english_market_name(market_name) + "\n(" + english_industry_name(industry_name) + ")"


def line_label(market_name, industry_name):
    return english_market_name(market_name) + " - " + english_industry_name(industry_name)


# Read the CSV file and combine quarterly records.
pair_data = {}
line_count = 0

with open(DATA_FILE, encoding="cp949") as fp:
    header = fp.readline()
    line = fp.readline()

    while line:
        row = line.strip()[1:-1].split('","')

        if len(row) >= 9:
            quarter_code = row[0]
            market_type = row[2]
            market_code = row[3]
            market_name = row[4]
            industry_code = row[5]
            industry_name = row[6]

            sales = int(row[7])
            transactions = int(row[8])
            quarter_number = int(quarter_code[-1])

            pair_key = market_type + "|" + market_code + "|" + industry_code

            if pair_key not in pair_data:
                pair_data[pair_key] = [
                    market_type,
                    market_code,
                    market_name,
                    industry_code,
                    industry_name,
                    0,
                    0,
                    0,
                    0,
                    0,
                    0,
                    0
                ]

            pair_data[pair_key][5] = pair_data[pair_key][5] + sales
            pair_data[pair_key][6] = pair_data[pair_key][6] + transactions
            pair_data[pair_key][6 + quarter_number] = pair_data[pair_key][6 + quarter_number] + sales
            pair_data[pair_key][11] = pair_data[pair_key][11] + 1

            line_count = line_count + 1

        line = fp.readline()

print("Data loading completed")
print("Number of rows:", line_count)
print("Number of market-industry combinations:", len(pair_data))


# Convert dictionary values to a list.
records = []

for pair_key in pair_data:
    records.append(pair_data[pair_key])


def top_records_in_type(records, target_type, number_of_records):
    selected = []

    for record in records:
        if record[0] == target_type:
            selected.append([
                record[5],
                record[6],
                record[0],
                record[2],
                record[4],
                record[7],
                record[8],
                record[9],
                record[10],
                record[11]
            ])

    selected.sort(reverse=True)
    return selected[:number_of_records]


# Print annual-sales top 5 by market type.
market_types = [
    "발달상권",
    "전통시장",
    "관광특구",
    "골목상권"
]

print("\nTop 5 Annual Sales by Market Type")

for market_type in market_types:
    top5 = top_records_in_type(records, market_type, 5)

    print("\n[" + market_type_en[market_type] + "]")

    rank = 1

    for item in top5:
        annual_sales_billion = item[0] / 1000000000.0

        print(
            str(rank)
            + ". "
            + english_market_name(item[3])
            + " / "
            + english_industry_name(item[4])
            + " | Annual Sales: "
            + str(round(annual_sales_billion, 1))
            + " billion KRW"
        )

        rank = rank + 1


# Calculate local concentration (LQ) and quarterly stability.
market_total = {}
industry_total = {}
type_total = {}

for record in records:
    market_key = record[0] + "|" + record[1]
    industry_key = record[0] + "|" + record[3]
    market_type = record[0]
    annual_sales = record[5]

    if market_key not in market_total:
        market_total[market_key] = 0

    if industry_key not in industry_total:
        industry_total[industry_key] = 0

    if market_type not in type_total:
        type_total[market_type] = 0

    market_total[market_key] = market_total[market_key] + annual_sales
    industry_total[industry_key] = industry_total[industry_key] + annual_sales
    type_total[market_type] = type_total[market_type] + annual_sales


sales_by_type = {}

for record in records:
    market_type = record[0]

    if market_type not in sales_by_type:
        sales_by_type[market_type] = []

    sales_by_type[market_type].append(record[5])


sales_threshold = {}

for market_type in sales_by_type:
    sales_by_type[market_type].sort()
    threshold_position = int(len(sales_by_type[market_type]) * 0.90)
    sales_threshold[market_type] = sales_by_type[market_type][threshold_position]


strong_pairs = []

for record in records:
    market_type = record[0]
    market_key = record[0] + "|" + record[1]
    industry_key = record[0] + "|" + record[3]

    annual_sales = record[5]

    q1 = record[7]
    q2 = record[8]
    q3 = record[9]
    q4 = record[10]

    number_of_quarters = record[11]

    market_share = annual_sales / market_total[market_key]
    industry_share = industry_total[industry_key] / type_total[market_type]
    lq = market_share / industry_share

    quarterly_average = (q1 + q2 + q3 + q4) / 4.0

    quarterly_variance = (
        (q1 - quarterly_average) ** 2
        + (q2 - quarterly_average) ** 2
        + (q3 - quarterly_average) ** 2
        + (q4 - quarterly_average) ** 2
    ) / 4.0

    quarterly_standard_deviation = quarterly_variance ** 0.5

    coefficient_of_variation = quarterly_standard_deviation / quarterly_average

    if q1 == 0:
        q1_to_q4_change = 0
    else:
        q1_to_q4_change = (q4 - q1) / q1

    if annual_sales >= sales_threshold[market_type] \
       and lq >= 1.5 \
       and number_of_quarters == 4 \
       and coefficient_of_variation <= 0.35:

        strong_pairs.append([
            annual_sales,
            lq,
            coefficient_of_variation,
            q1_to_q4_change,
            market_type,
            record[2],
            record[4]
        ])


print("\nStrong Market-Industry Pairs")
print("Criteria: high sales, high local concentration, and stable quarterly sales")

for market_type in market_types:
    selected = []

    for item in strong_pairs:
        if item[4] == market_type:
            selected.append(item)

    selected.sort(reverse=True)
    selected = selected[:10]

    print("\n[" + market_type_en[market_type] + "]")

    rank = 1

    for item in selected:
        annual_sales_billion = item[0] / 1000000000.0

        print(
            str(rank)
            + ". "
            + english_market_name(item[5])
            + " / "
            + english_industry_name(item[6])
            + " | Sales: "
            + str(round(annual_sales_billion, 1))
            + " billion KRW"
            + " | LQ: "
            + str(round(item[1], 2))
            + " | CV: "
            + str(round(item[2], 2))
            + " | 1st to 4th Quarter Change: "
            + str(round(item[3] * 100.0, 1))
            + "%"
        )

        rank = rank + 1


# Bar chart: Actual market and industry names.
selected_type = "발달상권"
top5 = top_records_in_type(records, selected_type, 5)

labels = []
values = []

for item in top5:
    labels.append(bar_label(item[3], item[4]))
    values.append(item[0] / 1000000000.0)

fig, ax = plt.subplots(figsize=(14, 6.5))
ax.bar(labels, values)

ax.set_title("Top 5 Annual Sales by Market and Industry", fontsize=14)
ax.set_xlabel("Commercial Area and Industry", fontsize=11)
ax.set_ylabel("2025 Cumulative Sales (billion KRW)", fontsize=11)

plt.tight_layout()
plt.show()


# Line chart: Actual market and industry names; full quarter names.
quarters = [
    "1st Quarter",
    "2nd Quarter",
    "3rd Quarter",
    "4th Quarter"
]

fig, ax = plt.subplots(figsize=(14, 7))

for item in top5:
    quarterly_sales = [
        item[5] / 1000000000.0,
        item[6] / 1000000000.0,
        item[7] / 1000000000.0,
        item[8] / 1000000000.0
    ]

    ax.plot(
        quarters,
        quarterly_sales,
        marker="o",
        label=line_label(item[3], item[4])
    )

ax.set_title("Quarterly Sales Trends of Top Commercial-Area Industries", fontsize=14)
ax.set_xlabel("Quarter", fontsize=11)
ax.set_ylabel("Sales (billion KRW)", fontsize=11)
ax.legend(loc="upper left", fontsize=9)

plt.tight_layout()
plt.show()
